# BP Data Preparation (Aligned Workflow)

## Notebook Contract
- Primary Yaw's input: `./Datasets/Raw Dataset/yaws_physical_properties_inhouse.csv` (in-house digitization; authoritative BP values and codes, with locally recovered SMILES)
- Adapter behavior: keep organic rows, use their SMILES directly, rename the BP fields, and retain only `SMILES`, `BP_K`, and `BP_code` before the existing pipeline
- The remaining cleaning, deduplication, filtering, and split logic is unchanged.


## 1. Setup and Load Raw Data


In [ ]:
import os
from collections import Counter
from pathlib import Path

import pandas as pd
from rdkit import Chem
from sklearn.model_selection import train_test_split

import sys
EXTERNAL_FUNCTIONS_DIR = (Path.cwd().parent / 'External Functions').resolve()
PLOTTING_FUNCTIONS_DIR = (EXTERNAL_FUNCTIONS_DIR / 'Plotting Functions').resolve()
for path in (EXTERNAL_FUNCTIONS_DIR, PLOTTING_FUNCTIONS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from dataprep_plotting import run_dataprep_plots

# Paths
CURATED_DIR = Path('./Datasets/Curated Dataset(s)')
RAW_DIR = Path('./Datasets/Raw Dataset')
RAW_BP_FILE = CURATED_DIR / 'Small_Organic_BPs.csv'
YAWS_BP_FILE = RAW_DIR / 'yaws_physical_properties_inhouse.csv'
NO_SPLIT_OUTPUT = CURATED_DIR / 'Small_Organic_SMILES_to_BP_no_splits.csv'
TRAINABLE_DIR = Path('./Datasets/trainable_data')
TRAINABLE_OUTPUT = TRAINABLE_DIR / 'BP_chemprop_train_file.csv'

# Cleaning controls
RANDOM_SEED = 42

# Known-bad datapoints to exclude (canonical SMILES)
EXCLUDE_SMILES = {'CNS(C)(=O)=O'}
RARE_ELEMENT_THRESHOLD = 0.01
MAX_HEAVY_ATOMS = 40

# Boiling-point unit normalization controls
# Set source units as 'K' or 'C'. All values are converted to Kelvin downstream.
SMALL_BP_UNIT = 'K'
YAWS_BP_UNIT = 'K'

# If a source is marked as Kelvin, optionally repair obviously Celsius-like negatives.
AUTO_CONVERT_NEGATIVE_BP_TO_K = True

# Yaw's BP code inclusion filter (keep only these normalized code values)
# Normalization removes spaces and parentheses, so '(1,2)' -> '1,2'.
YAWS_INCLUDED_BP_CODES = {'1', '1,2', '2'}


def normalize_bp_to_kelvin(series, unit='K', auto_convert_negative=False):
    values = pd.to_numeric(series, errors='coerce').copy()
    converted_count = 0
    unit_norm = str(unit).strip().upper()
    if unit_norm == 'C':
        mask = values.notna()
        values.loc[mask] = values.loc[mask] + 273.15
        converted_count = int(mask.sum())
    elif unit_norm == 'K':
        if auto_convert_negative:
            neg_mask = values < 0
            values.loc[neg_mask] = values.loc[neg_mask] + 273.15
            converted_count = int(neg_mask.sum())
    else:
        raise ValueError(f"Unsupported unit '{unit}'. Use 'K' or 'C'.")
    return values, converted_count


def canonicalize_smiles_for_merge(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None


def _pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


# Load primary BP dataset (preferred source for duplicates)
Small_Organic_BP_df = pd.read_csv(RAW_BP_FILE)
small_rows_raw = len(Small_Organic_BP_df)

# Load the in-house Yaw's table, including the locally recovered SMILES column.
yaws_source_df = pd.read_csv(YAWS_BP_FILE, dtype=str, keep_default_na=False)
required_yaws_source_cols = {
    'section', 'SMILES', 'boiling_point_K', 'boiling_point_code'
}
missing_yaws_source = required_yaws_source_cols - set(yaws_source_df.columns)
if missing_yaws_source:
    raise ValueError(f"Missing required columns in in-house Yaw's file: {sorted(missing_yaws_source)}")

yaws_raw_df = yaws_source_df.loc[
    yaws_source_df['section'].eq('organic'),
    ['SMILES', 'boiling_point_K', 'boiling_point_code']
].rename(columns={
    'boiling_point_K': 'BP_K',
    'boiling_point_code': 'BP_code',
})
yaws_raw_df = yaws_raw_df.loc[yaws_raw_df['SMILES'].astype(str).str.strip().ne('')].copy()
yaws_rows_raw = len(yaws_raw_df)

small_smiles_col = _pick_col(Small_Organic_BP_df, ['SMILES', 'smiles'])
small_bp_col = _pick_col(Small_Organic_BP_df, ['BP', 'bp', 'BP_K', 'bp_k', 'Boiling_Point', 'boiling_point'])
yaws_smiles_col = _pick_col(yaws_raw_df, ['SMILES', 'smiles'])
yaws_bp_col = _pick_col(yaws_raw_df, ['BP', 'bp', 'BP_K', 'bp_k', 'Boiling_Point', 'boiling_point'])

if small_smiles_col is None or small_bp_col is None:
    raise ValueError('Small_Organic_BPs.csv must contain SMILES and BP columns.')
if yaws_smiles_col is None or yaws_bp_col is None:
    raise ValueError("Adapted in-house Yaw's data must contain SMILES and BP columns.")

small_df = Small_Organic_BP_df[[small_smiles_col, small_bp_col]].copy()
small_df.columns = ['SMILES', 'BP']
small_df['BP'], small_bp_unit_fix_count = normalize_bp_to_kelvin(
    small_df['BP'], unit=SMALL_BP_UNIT, auto_convert_negative=AUTO_CONVERT_NEGATIVE_BP_TO_K
)
small_df['source_dataset'] = 'Small_Organic_BPs'

yaws_df = yaws_raw_df.copy()
yaws_code_col = _pick_col(yaws_df, ['BP_Code', 'BP code', 'BP_code', 'BPCode'])
if yaws_code_col is not None:
    yaws_code_norm = (
        yaws_df[yaws_code_col]
        .astype(str)
        .str.replace(' ', '', regex=False)
        .str.replace('(', '', regex=False)
        .str.replace(')', '', regex=False)
    )
    yaws_code_targets = {'(1)': '1', '(1,2)': '1,2', '(2)': '2', '(2,3)': '2,3', '(3)': '3'}
    print("Yaw's BP code counts (pre-filter):")
    for label, key in yaws_code_targets.items():
        print(f"  {label}: {(yaws_code_norm == key).sum()}")
if yaws_code_col is not None:
    yaws_df = yaws_df[yaws_code_norm.isin(YAWS_INCLUDED_BP_CODES)].copy()

yaws_df = yaws_df[[yaws_smiles_col, yaws_bp_col]].copy()
yaws_df.columns = ['SMILES', 'BP']
yaws_df['BP'], yaws_bp_unit_fix_count = normalize_bp_to_kelvin(
    yaws_df['BP'], unit=YAWS_BP_UNIT, auto_convert_negative=AUTO_CONVERT_NEGATIVE_BP_TO_K
)
yaws_df['source_dataset'] = 'yaws_bp_dataset'

small_df['SMILES'] = small_df['SMILES'].apply(canonicalize_smiles_for_merge)
yaws_df['SMILES'] = yaws_df['SMILES'].apply(canonicalize_smiles_for_merge)

small_df = small_df.dropna(subset=['SMILES', 'BP']).copy()
yaws_df = yaws_df.dropna(subset=['SMILES', 'BP']).copy()

combined_bp_df = pd.concat([small_df, yaws_df], ignore_index=True)

print(f'Small_Organic_BPs raw rows: {small_rows_raw}')
print(f"In-house Yaw's rows with SMILES: {yaws_rows_raw}")
print(f'Small dataset unit repairs applied: {small_bp_unit_fix_count}')
print(f'Yaws dataset unit repairs applied: {yaws_bp_unit_fix_count}')
print(f'Combined rows before cleaning: {len(combined_bp_df)}')


## 2. Data Filtering


In [ ]:
def clean_smiles_dataset(
    df,
    output_path,
    threshold=0.01,
    max_heavy_atoms=40,
):
    smiles_col = None
    for candidate in ['SMILES', 'smiles']:
        if candidate in df.columns:
            smiles_col = candidate
            break
    if smiles_col is None:
        raise ValueError("No 'SMILES' or 'smiles' column found in DataFrame.")

    def canonicalize_smiles_inner(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None:
                return None
            return Chem.MolToSmiles(mol, canonical=True)
        except Exception:
            return None

    def is_organic(mol):
        return any(atom.GetSymbol() == 'C' for atom in mol.GetAtoms())

    def is_organometallic(mol):
        metal_atomic_numbers = {
            3,4,11,12,13,14,19,20,21,22,23,24,25,26,27,28,29,30,
            31,37,38,39,40,41,42,43,44,45,46,47,48,49,50,
            55,56,57,72,73,74,75,76,77,78,79,80,81,82,83,
            87,88,89
        }
        return any(atom.GetAtomicNum() in metal_atomic_numbers for atom in mol.GetAtoms())

    def is_ion(mol):
        return Chem.GetFormalCharge(mol) != 0

    def has_isotopic_label(mol):
        # Remove explicitly isotopically labeled compounds such as deuterated SMILES ([2H]).
        return any(atom.GetIsotope() > 0 for atom in mol.GetAtoms())


    def is_radical(mol):
        return any(atom.GetNumRadicalElectrons() > 0 for atom in mol.GetAtoms())

    def count_heavy_atoms(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return mol.GetNumHeavyAtoms()

    def analyze_elements(df_local, smiles_column):
        atom_type_counts = Counter()
        molecule_atom_presence = Counter()
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            mol = Chem.AddHs(mol)
            atom_symbols = [atom.GetSymbol() for atom in mol.GetAtoms()]
            atom_type_counts.update(atom_symbols)
            molecule_atom_presence.update(set(atom_symbols))
        return pd.DataFrame({
            'Total_Atom_Count': pd.Series(atom_type_counts),
            'Molecule_Count': pd.Series(molecule_atom_presence)
        }).fillna(0).astype(int)

    def remove_rare_elements(df_local, smiles_column, threshold_local):
        atom_summary_local = analyze_elements(df_local, smiles_column=smiles_column)
        total_molecules = len(df_local)
        if total_molecules == 0:
            return df_local.copy(), atom_summary_local, [], 0

        rare_elements = atom_summary_local[
            atom_summary_local['Molecule_Count'] / total_molecules < threshold_local
        ].index.tolist()

        mask = []
        for smi in df_local[smiles_column]:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                mask.append(False)
                continue
            atoms = {atom.GetSymbol() for atom in mol.GetAtoms()}
            mask.append(not any(el in atoms for el in rare_elements))

        filtered_df = df_local[mask].reset_index(drop=True)
        removed_count = len(df_local) - len(filtered_df)
        return filtered_df, atom_summary_local, rare_elements, removed_count

    stats = {
        'invalid': [],
        'multi_component': [],
        'inorganic': [],
        'organometallic': [],
        'ion': [],
        'isotopic_label': [],
        'radical': [],
        'kept': [],
    }

    for _, row in df.iterrows():
        row_copy = row.copy()
        smi = row_copy[smiles_col]
        cano = canonicalize_smiles_inner(smi)
        if cano is None:
            stats['invalid'].append(row_copy)
            continue

        # Filter out multi-component SMILES (e.g., salts/mixtures like 'CCO.Cl')
        if '.' in str(cano):
            row_copy[smiles_col] = cano
            stats['multi_component'].append(row_copy)
            continue

        mol = Chem.MolFromSmiles(cano)
        if mol is None:
            stats['invalid'].append(row_copy)
            continue

        row_copy[smiles_col] = cano

        if not is_organic(mol):
            stats['inorganic'].append(row_copy)
            continue
        if is_organometallic(mol):
            stats['organometallic'].append(row_copy)
            continue
        if is_ion(mol):
            stats['ion'].append(row_copy)
            continue
        if has_isotopic_label(mol):
            stats['isotopic_label'].append(row_copy)
            continue
        if is_radical(mol):
            stats['radical'].append(row_copy)
            continue

        stats['kept'].append(row_copy)

    pre_dedup_df = pd.DataFrame(stats['kept'])
    pre_dedup_count = len(pre_dedup_df)
    dedup_df = pre_dedup_df.drop_duplicates(subset=[smiles_col], keep='first')
    dedup_removed = pre_dedup_count - len(dedup_df)

    cleaned_df, atom_summary, rare_elements, rare_removed = remove_rare_elements(dedup_df, smiles_col, threshold)

    cleaned_df['non_H_atom_count'] = cleaned_df[smiles_col].apply(count_heavy_atoms)
    heavy_before = len(cleaned_df)
    cleaned_df = cleaned_df[cleaned_df['non_H_atom_count'] <= max_heavy_atoms].copy()
    heavy_removed = heavy_before - len(cleaned_df)

    cleaned_df = cleaned_df.reset_index(drop=True)

    os.makedirs(output_path, exist_ok=True)
    output_file = Path(output_path) / 'Small_Organic_BP_clean.csv'
    cleaned_df.to_csv(output_file, index=False)
    print(f'Cleaned dataset saved to {output_file}')

    print('Filter breakdown (stepwise):')
    running = len(df)

    def print_step(label, removed):
        nonlocal running
        after = running - removed
        print(f'- {label}: removed {removed} | before {running} -> after {after}')
        running = after

    print_step('Invalid SMILES', len(stats['invalid']))
    print_step('Multi-component SMILES (contains .)', len(stats['multi_component']))
    print_step('Inorganic (no carbon)', len(stats['inorganic']))
    print_step('Organometallic', len(stats['organometallic']))
    print_step('Ions (non-zero formal charge)', len(stats['ion']))
    print_step('Isotopically labeled SMILES', len(stats['isotopic_label']))
    print_step('Radicals', len(stats['radical']))
    print_step('Duplicate SMILES', dedup_removed)
    print_step(f'Rare elements (<{threshold*100:.1f}% molecule frequency)', rare_removed)
    print_step(f'Heavy atom count > {max_heavy_atoms}', heavy_removed)

    if rare_elements:
        print(f'Rare elements identified: {rare_elements}')
    else:
        print('Rare elements identified: none')

    print(f'Final retained rows: {len(cleaned_df)}')

    filter_report = {
        'invalid': len(stats['invalid']),
        'multi_component': len(stats['multi_component']),
        'inorganic': len(stats['inorganic']),
        'organometallic': len(stats['organometallic']),
        'ion': len(stats['ion']),
        'isotopic_label': len(stats['isotopic_label']),
        'radical': len(stats['radical']),
        'deduplicated': dedup_removed,
        'rare_elements_removed': rare_removed,
        'heavy_atom_removed': heavy_removed,
    }

    return cleaned_df, atom_summary, filter_report


In [ ]:
Small_Organic_BP_df_element_filter, atom_summary, filter_report = clean_smiles_dataset(
    combined_bp_df,
    output_path=CURATED_DIR,
    threshold=RARE_ELEMENT_THRESHOLD,
    max_heavy_atoms=MAX_HEAVY_ATOMS,
)

# Drop any known-bad datapoints (if present)
exclude_set = EXCLUDE_SMILES if 'EXCLUDE_SMILES' in globals() else set()
if exclude_set:
    smiles_col = 'SMILES' if 'SMILES' in Small_Organic_BP_df_element_filter.columns else 'smiles'
    before = len(Small_Organic_BP_df_element_filter)
    Small_Organic_BP_df_element_filter = Small_Organic_BP_df_element_filter[
        ~Small_Organic_BP_df_element_filter[smiles_col].isin(exclude_set)
    ].copy()
    removed = before - len(Small_Organic_BP_df_element_filter)
    if removed:
        print(f'Removed known-bad datapoints: {removed}')

print('\nPost-cleaning shape:', Small_Organic_BP_df_element_filter.shape)
if 'source_dataset' in Small_Organic_BP_df_element_filter.columns:
    print('Rows by source after cleaning:')
    print(Small_Organic_BP_df_element_filter['source_dataset'].value_counts())


## 3. Diagnostics and Functional Group Profiles


In [ ]:
plot_df = None
for candidate in ('Small_Organic_BP_df_element_filter', 'cleaned_df', 'filtered_df', 'dedup_df'):
    if candidate in globals() and globals()[candidate] is not None:
        plot_df = globals()[candidate]
        break

if plot_df is None:
    raise NameError(
        "No cleaned BP dataframe found. Run the cleaning/prep cell first (the cell that calls clean_smiles_dataset)."
    )

run_dataprep_plots(
    plot_df,
    smiles_col='SMILES',
    property_col='BP',
    non_h_atom_col='non_H_atom_count',
    property_label='Boiling Point',
    property_units='K',
    property_hist_bins=50,
    top_n=30,
    max_per_row=6,
    include_reactive_group_distribution=True,
)


## 4. Export Curated BP Dataset (No Splits)


In [ ]:
df_subset = Small_Organic_BP_df_element_filter[['SMILES', 'BP']].copy()
df_subset.to_csv(NO_SPLIT_OUTPUT, index=False)
print(f'Wrote no-split curated file to: {NO_SPLIT_OUTPUT}')
print(f'Rows exported: {len(df_subset)}')


## 5. Random Split for Chemprop and Export Trainable File


In [ ]:
df_split = pd.read_csv(NO_SPLIT_OUTPUT)


def canonical_smiles(smiles: str) -> str:
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return Chem.MolToSmiles(mol, canonical=True)
    return smiles


# Grouped random split: ensures the same molecule never appears in multiple splits
df_split['_mol_id'] = df_split['SMILES'].astype(str).apply(canonical_smiles)
unique_ids = df_split['_mol_id'].drop_duplicates()

train_ids, temp_ids = train_test_split(unique_ids, test_size=0.2, random_state=RANDOM_SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=RANDOM_SEED)

df_split['split'] = None
df_split.loc[df_split['_mol_id'].isin(train_ids), 'split'] = 'train'
df_split.loc[df_split['_mol_id'].isin(val_ids), 'split'] = 'val'
df_split.loc[df_split['_mol_id'].isin(test_ids), 'split'] = 'test'

if df_split['split'].isna().any():
    raise ValueError(f"Unassigned split rows: {df_split['split'].isna().sum()}")

split_per_mol = df_split.groupby('_mol_id')['split'].nunique()
if not (split_per_mol == 1).all():
    bad = split_per_mol[split_per_mol != 1]
    raise ValueError(f"Molecules assigned to multiple splits: {len(bad)}")

df_split = df_split.drop(columns=['_mol_id'])

TRAINABLE_DIR.mkdir(parents=True, exist_ok=True)
df_split.to_csv(TRAINABLE_OUTPUT, index=False)

print(f'Wrote Chemprop train file to: {TRAINABLE_OUTPUT}')
print(df_split['split'].value_counts(dropna=False))


## 6. Final Summary


In [ ]:
print('Final summary:')
print(f'- Small_Organic_BPs raw rows: {small_rows_raw}')
print(f'- yaws_bp_dataset raw rows: {yaws_rows_raw}')
print(f'- Combined rows before cleaning: {len(combined_bp_df)}')
print(f'- Cleaned rows: {len(Small_Organic_BP_df_element_filter)}')
print(f'- No-split export: {NO_SPLIT_OUTPUT}')
print(f'- Trainable export: {TRAINABLE_OUTPUT}')
print('- Filter report:')
for key, value in filter_report.items():
    print(f'  - {key}: {value}')
